# R.U.Psycho quickstart

This notebook walks through the whole workflow on your machine, **offline and in a few
seconds**: load a questionnaire experiment, look at the prompt the model will see, add a
model, run it, collect and post-process the answers, and check that the results are
reproducible.

No model is downloaded and no API key is needed. The model used here is a tiny
**scripted stand-in** that is defined in this notebook and clearly marked as such.
Everything else (configuration, run loop, callbacks, seeding, parsers, post-processing)
is the real package, and swapping the stand-in for a real model is a one-line change.

1. Installation
2. The bundled example
3. Inspect the assembled prompt
4. Add a model
5. Run it with a callback
6. The `RunSummary` and the answers table
7. Failing calls: the `on_error` policy
8. Concurrency for API models
9. Reproducibility: seeds
10. Post-processing: clean, validate, judge
11. From answers to scores
12. The command line

## 1. Installation

```bash
pip install git+https://github.com/julianschelb/rupsycho.git
```

Model back-ends are optional extras, so the core install stays light:

| Extra | Adds | Needed for |
| --- | --- | --- |
| `huggingface` | torch, transformers, langchain-huggingface | local models (`local_huggingface`) and the model-based parsers |
| `openai` | langchain-openai | OpenAI and OpenAI-compatible endpoints |
| `ollama` | langchain-ollama | models served by Ollama |
| `google` | langchain-google-genai | Google Gemini |
| `deepseek` | langchain-deepseek | DeepSeek |
| `models` | all of the above | every back-end |
| `notebook` | ipywidgets | progress bars in Jupyter |

```bash
pip install "rupsycho[huggingface,notebook] @ git+https://github.com/julianschelb/rupsycho.git"
```

This notebook needs **none** of the extras. A missing extra is reported with the exact
`pip install` command as soon as a feature needs it.

## 2. The bundled example

The package ships a small Big Five Inventory (BFI) experiment, so there is something to
run right after installing it. An experiment is one declarative description of a study:
the questionnaire, the personas that answer it, the models, the prompt template and the
random seeds.

`models={}` drops the Hugging Face model that the example comes with, because we bring
our own model below.

In [1]:
import rupsycho as rup

rup.list_examples()

['bfi']

In [2]:
experiment = rup.load_example_experiment("bfi", models={})

print("experiment:", experiment.name)
print("personas  :", experiment.list_personas())
print("questions :", experiment.questionnaire.get_number_of_questions())
print("seeds     :", experiment.parameters.seeds)
print("models    :", experiment.list_models())

experiment: Big Five Inventory example
personas  : ['Optimistic Persona', 'Conservative Persona']
questions : 5
seeds     : ['7']
models    : []


The questionnaire defines the answer options. Their `weight` is what turns a chosen
option into a score later on (section 11).

In [3]:
for option in experiment.questionnaire.default_answer_options.options.values():
    print(f"weight {option.weight}: {option.text}")

weight 1: 1. Disagree strongly
weight 2: 2. Disagree a little
weight 3: 3. Neither agree nor disagree
weight 4: 4. Agree a little
weight 5: 5. Agree strongly


## 3. Inspect the assembled prompt

Always read the prompt before you spend compute: it is the persona description, the
general instruction, the question and the answer options poured into the template.
Use `persona_idx=1` for the other persona.

In [4]:
experiment.print_assembled_prompt(item_idx=0, persona_idx=0)


++++++++++++++++++ assembled prompt (item 0, persona 0) ++++++++++++++++++
System: Objective: Answer as the following survey participant: Ms Muller is 18 years old and very open minded with an optimistic personality.
Here are a number of characteristics that may or may not apply to you. Please indicate the extent to which you agree or disagree with each statement by choosing one of the answer options.
Instructions: Choose from the list of answer options to answer the question. Answer the question using only the provided answer options. If none of the options are correct, choose the option that is closest to being correct. The solution must be provided in this format: {answer: "answer option"}
Human: Question: I see myself as someone who is talkative.
Answer Options: 1. Disagree strongly, 2. Disagree a little, 3. Neither agree nor disagree, 4. Agree a little, 5. Agree strongly
Answer:
+++++++++++++++++++++++++++++++++++++++++++++++++++++


## 4. Add a model

An experiment asks every **model** every **question** as every **persona**, once per
**seed**. Models are either part of the configuration (`"models": {...}` sections of
type `local_huggingface`, `openai`, `ollama`, `google`, `deepseek`, ...) or added in code
with `experiment.add_model(model, identifier=...)`, where `model` is any LangChain model.

The next cell defines `ScriptedLLM`. **It is a stand-in for a real model, not a language
model**: it answers with an option number that is a pseudo-random function of the seed
and the prompt, just like an API model that is called with a `seed`. Its switches let us
show free-text answers, latency and failures later on. Nothing is downloaded.

In [5]:
import random
import time
from typing import Any

from langchain_core.language_models.llms import LLM

LABELS = {
    1: "Disagree strongly",
    2: "Disagree a little",
    3: "Neither agree nor disagree",
    4: "Agree a little",
    5: "Agree strongly",
}
# Free-text replies of the "chatty" mode (used for the post-processing in section 10)
REPLIES = [
    "{n}",
    "{n}. {label}",
    "I would say {label}.",
    '{{answer: "{n}. {label}"}}',
    "Hmm, let me think about it...\n\n{label}. \N{SMILING FACE WITH SMILING EYES}",
    "As an AI, I do not have personal opinions.",
    "Sorry, I can't answer that.",
    "It depends on the situation.",
]


class ScriptedLLM(LLM):
    """STAND-IN FOR A REAL MODEL: answers with an option number, offline and deterministic."""

    seed: int | None = None  # set by rupsycho for every run, see section 9
    chatty: bool = False  # answer in free text instead of a bare number
    latency: float = 0.0  # seconds to wait per call, to imitate a remote API
    fail_on: str | None = None  # raise an error for prompts that contain this text

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _call(
        self,
        prompt: str,
        stop: list[str] | None = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> str:
        if self.latency:
            time.sleep(self.latency)
        if self.fail_on and self.fail_on in prompt:
            raise RuntimeError(f"cannot answer prompts containing {self.fail_on!r}")
        # Same seed and prompt -> same draw. Without a seed every call is a fresh sample.
        seed = self.seed
        rng = random.SystemRandom() if seed is None else random.Random(f"{seed}|{prompt}")
        number = rng.randint(1, 5)
        if not self.chatty:
            return str(number)
        return rng.choice(REPLIES).format(n=number, label=LABELS[number])


def new_experiment(model, identifier="scripted", seeds=None):
    """The bundled BFI example with `model` as its only model (and optionally new seeds)."""
    config = rup.load_example_config("bfi")  # a plain dict that you may edit freely
    config["models"] = {}
    if seeds:
        config["parameters"]["seeds"] = seeds
    experiment = rup.experiment_from_dict(config)
    experiment.add_model(model, identifier=identifier)
    return experiment

In [6]:
experiment.add_model(ScriptedLLM(), identifier="scripted")
experiment.list_models()

['scripted']

With a real model this is the only line that changes (needs the matching extra and, for
hosted models, an API key in the environment):

```python
from langchain_openai import ChatOpenAI

experiment.add_model(ChatOpenAI(model="gpt-4o-mini", temperature=0.7), identifier="gpt-4o-mini")
```

or keep the model of the example and let rupsycho load it when the run reaches it:
`rup.load_example_experiment("bfi")`.

## 5. Run it with a callback

`run()` asks every question and hands each answer to the callbacks **as soon as it is
generated**, so partial results survive a crash. `CSVCallback` appends one row per
answer to a CSV file; we write into a scratch folder that is deleted at the end.
A progress bar is shown by default (in Jupyter it needs the `notebook` extra);
`show_progress=False` keeps this notebook tidy.

In [7]:
import tempfile
from pathlib import Path

from rupsycho.callbacks import CSVCallback

workdir = Path(tempfile.mkdtemp(prefix="rupsycho-quickstart-"))  # removed at the end
csv_path = workdir / "answers.csv"

summary = experiment.run(callbacks=[CSVCallback(csv_path)], show_progress=False)

## 6. The `RunSummary` and the answers table

`run()` returns a `RunSummary`: how many model calls were made, how many failed and the
first distinct error messages. `summary.elapsed` holds the wall-clock seconds; it is not
printed here to keep the output of this notebook stable (`str(summary)` reads like
`8 model calls in 0.1s`).

In [8]:
print("model calls:", summary.n_calls)
print("succeeded  :", summary.n_succeeded)
print("failed     :", summary.n_failed)
print("errors     :", summary.errors)

model calls: 10
succeeded  : 10
failed     : 0
errors     : []


The answers are also kept on the experiment. `get_answers_as_dataframe()` returns one
row per model x persona x seed x question:

In [9]:
answers = experiment.get_answers_as_dataframe()
answers

,Instruction ID,Instruction Question,Model ID,Persona ID,Run Seed,Answer
0,0,I see myself as someone who is talkative.,scripted,Optimistic Persona,7,2
1,0,I see myself as someone who is talkative.,scripted,Conservative Persona,7,1
2,1,I see myself as someone who tends to find faul...,scripted,Optimistic Persona,7,5
3,1,I see myself as someone who tends to find faul...,scripted,Conservative Persona,7,3
4,2,I see myself as someone who does a thorough job.,scripted,Optimistic Persona,7,3
5,2,I see myself as someone who does a thorough job.,scripted,Conservative Persona,7,1
6,3,"I see myself as someone who is depressed, blue.",scripted,Optimistic Persona,7,1
7,3,"I see myself as someone who is depressed, blue.",scripted,Conservative Persona,7,3
8,4,"I see myself as someone who is original, comes...",scripted,Optimistic Persona,7,2
9,4,"I see myself as someone who is original, comes...",scripted,Conservative Persona,7,5


The callback wrote the same answers to disk, plus the time each call took (dropped
below, because it differs from run to run). Note that pandas reads numeric answers as
numbers; the post-processing pipeline takes care of that.

In [10]:
import pandas as pd

saved = pd.read_csv(csv_path)
print(list(saved.columns))
saved.drop(columns=["experiment_name", "instruction_item", "time"])

['experiment_name', 'instruction_item_id', 'instruction_item', 'model_id', 'profile_id', 'random_seed', 'time', 'answer']


,instruction_item_id,model_id,profile_id,random_seed,answer
0,0,scripted,Optimistic Persona,7,2
1,0,scripted,Conservative Persona,7,1
2,1,scripted,Optimistic Persona,7,5
3,1,scripted,Conservative Persona,7,3
4,2,scripted,Optimistic Persona,7,3
5,2,scripted,Conservative Persona,7,1
6,3,scripted,Optimistic Persona,7,1
7,3,scripted,Conservative Persona,7,3
8,4,scripted,Optimistic Persona,7,2
9,4,scripted,Conservative Persona,7,5


## 7. Failing calls: the `on_error` policy

Remote models time out, hit rate limits or refuse. A failed call never takes the whole
experiment down; what happens instead is the `on_error` policy of `run()`:

| `on_error` | Behaviour |
| --- | --- |
| `"warn"` (default) | log every failed call, continue, warn once at the end |
| `"raise"` | stop at the first failure and raise its exception |
| `"ignore"` | continue silently; inspect the `RunSummary` |

Failed calls have no answer, so they are missing from the table. Callbacks receive them
with `answer=None`. Here the scripted model fails for the second persona on purpose:

In [11]:
import warnings

failing = new_experiment(ScriptedLLM(fail_on="Grueber"))  # the second persona is "Mr Grueber"

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    summary = failing.run(show_progress=False)  # on_error="warn"

print(f"{summary.n_failed} of {summary.n_calls} calls failed")
print("errors :", summary.errors)
print("warning:", caught[0].message)
failing.get_answers_as_dataframe()["Persona ID"].value_counts()

Model call failed (model=scripted, persona=Conservative Persona, item=0, seed=7): RuntimeError: cannot answer prompts containing 'Grueber'
Model call failed (model=scripted, persona=Conservative Persona, item=1, seed=7): RuntimeError: cannot answer prompts containing 'Grueber'
Model call failed (model=scripted, persona=Conservative Persona, item=2, seed=7): RuntimeError: cannot answer prompts containing 'Grueber'
Model call failed (model=scripted, persona=Conservative Persona, item=3, seed=7): RuntimeError: cannot answer prompts containing 'Grueber'
Model call failed (model=scripted, persona=Conservative Persona, item=4, seed=7): RuntimeError: cannot answer prompts containing 'Grueber'


5 of 10 calls failed
errors : ["RuntimeError: cannot answer prompts containing 'Grueber'"]


Persona ID
Optimistic Persona    5
Name: count, dtype: int64

In [12]:
summary = failing.run(on_error="ignore", show_progress=False)
print("ignored failures:", summary.n_failed)

try:
    failing.run(on_error="raise", show_progress=False)
except RuntimeError as error:
    print("stopped:", error)

ignored failures: 5
stopped: cannot answer prompts containing 'Grueber'


## 8. Concurrency for API models

For hosted models the time is spent waiting for the server, so `max_concurrency=N` runs
up to `N` calls at the same time. Results and callbacks keep their order, and a failing
call does not affect the others. Models that run inside this Python process (local
Hugging Face pipelines) share global state and are always called one after the other,
with a warning; `rupsycho.seeding.is_thread_safe(model)` tells which kind a model is.

In [13]:
def timed_run(max_concurrency):
    experiment = new_experiment(ScriptedLLM(latency=0.2))  # every call "takes" 0.2 s
    started = time.perf_counter()
    experiment.run(max_concurrency=max_concurrency, show_progress=False)
    return experiment.get_answers_as_dataframe(), time.perf_counter() - started


sequential, sequential_seconds = timed_run(1)
concurrent, concurrent_seconds = timed_run(8)

print("identical answers, same order    :", sequential.equals(concurrent))
print("concurrent run at least 2x faster:", sequential_seconds > 2 * concurrent_seconds)

identical answers, same order    : True
concurrent run at least 2x faster: True


## 9. Reproducibility: seeds

`parameters.seeds` lists the seeds of the experiment, and every seed is a full repetition
of the study. The seed is handed to the model by `rupsycho.seeding`, which knows how
each back-end wants it:

| Back-end | How the seed is applied |
| --- | --- |
| local Hugging Face pipelines | `transformers.set_seed(seed)` right before every call |
| OpenAI, DeepSeek, Ollama, ... | the model's `seed` parameter (a copy of the model with the seed set) |
| Hugging Face endpoints | the `seed` parameter of the request |
| Google Gemini and other back-ends without seed support | cannot be seeded: a warning is emitted |

Our stand-in has a `seed` field like the API models, so `seed_model` can make it
reproducible:

In [14]:
from rupsycho import seeding

model = ScriptedLLM()
seeded = seeding.seed_model(model, 42)
print("supports seeding:", seeding.supports_seeding(model))
print("seed_model returns a", type(seeded).__name__, "with seed", seeded.seed)

supports seeding: True
seed_model returns a ScriptedLLM with seed 42


In [15]:
def answers_for(seeds):
    experiment = new_experiment(ScriptedLLM(), seeds=seeds)
    experiment.run(show_progress=False)
    return experiment.get_answers_as_dataframe()["Answer"].tolist()


first, again, other = answers_for(["1", "2"]), answers_for(["1", "2"]), answers_for(["3", "4"])

print("same seeds      -> identical answers :", first == again)
print("different seeds -> different answers :", first != other)

same seeds      -> identical answers : True
different seeds -> different answers : True


A model that cannot be seeded is detected, and you are told instead of silently getting
non-reproducible results. A custom strategy can be registered with
`rupsycho.seeding.register_seeder(ModelClass, seeder)`.

In [16]:
class CoinFlipLLM(LLM):
    """STAND-IN for a back-end without seed support (such as Google Gemini)."""

    @property
    def _llm_type(self) -> str:
        return "coin-flip"

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        return str(random.randint(1, 5))  # global random state: rupsycho cannot control it


def coin_flip_answers():
    experiment = new_experiment(CoinFlipLLM(), identifier="coin-flip")
    experiment.run(show_progress=False)
    return experiment.get_answers_as_dataframe()["Answer"].tolist()


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    run_1 = coin_flip_answers()
run_2 = coin_flip_answers()

print("supports seeding :", seeding.supports_seeding(CoinFlipLLM()))
print("warning          :", caught[0].message)
print("identical answers:", run_1 == run_2)

supports seeding : False
identical answers: False


Real back-ends can still differ between hardware, library versions or model revisions
even when the seed is honoured; pin model revisions in the configuration
(`revision` of local models) and keep the exported experiment next to your results.
`examples/reproducibility_check.py` runs the same checks, and one more with a tiny
local Hugging Face model that it builds on the fly.

## 10. Post-processing: clean, validate, judge

Models answer in free text ("Hmm, let me think about it... Agree a little."), so the raw
answers need three rule-based steps before they can be scored:

1. a **cleaner** removes line breaks, emoji and other noise (`BasicCleaner`),
2. a **validator** flags answers that are no answers, such as refusals or "As an AI ..."
   (`ValidatorParser`),
3. a **judge** decides which answer option the text means (`MultipleChoiceJudge`).

All of them are LangChain output parsers. First on single strings:

In [17]:
from rupsycho.parsers.cleaners import BasicCleaner
from rupsycho.parsers.judges import MultipleChoiceJudge
from rupsycho.parsers.validators import ValidatorParser

options = experiment.questionnaire.default_answer_options.get_options_as_list()
cleaner, validator, judge = BasicCleaner(), ValidatorParser(), MultipleChoiceJudge(options)

samples = [
    "4",
    "Hmm, let me think about it...\n\nAgree a little. \N{SMILING FACE WITH SMILING EYES}",
    '{answer: "2. Disagree a little"}',
    "As an AI, I do not have personal opinions.",
    "It depends on the situation.",
    "Somewhere between 2 and 3.",
]
rows = []
for raw in samples:
    cleaned = cleaner.parse(raw)
    valid = validator.parse(cleaned)["validation_status"] == "valid"
    rows.append((raw, cleaned, valid, judge.parse(cleaned)))
pd.DataFrame(rows, columns=["raw", "cleaned", "valid", "decision"])

,raw,cleaned,valid,decision
0,4,4,True,4. Agree a little
1,"Hmm, let me think about it...\n\nAgree a littl...","Hmm, let me think about it... Agree a little.",True,4. Agree a little
2,"{answer: ""2. Disagree a little""}","{answer: ""2. Disagree a little""}",True,2. Disagree a little
3,"As an AI, I do not have personal opinions.","As an AI, I do not have personal opinions.",False,not present
4,It depends on the situation.,It depends on the situation.,True,not present
5,Somewhere between 2 and 3.,Somewhere between 2 and 3.,True,inconclusive


The judge answers `not present` when no option is recognisable and `inconclusive` when
several fit equally well. A refusal such as the fourth sample is flagged by the validator
and has to be excluded from the scores, whatever the judge says.

For whole result files the `PostprocessingPipeline` applies the same three steps to the
CSV written by `CSVCallback`. It reads the questionnaire from the configuration file,
so we first save the configuration. To get a realistic mix of answers, the chatty
stand-in answers the study three times (seeds 1 to 3):

In [18]:
import json

from rupsycho.postprocessing import PostprocessingPipeline

config_path = workdir / "config.json"
config_path.write_text(json.dumps(rup.load_example_config("bfi")), encoding="utf-8")

chatty = new_experiment(ScriptedLLM(chatty=True), seeds=["1", "2", "3"])
chatty.run(callbacks=[CSVCallback(workdir / "chatty.csv")], show_progress=False)

pipeline = PostprocessingPipeline(
    config_file_path=config_path,
    results_file_patterns=str(workdir / "chatty.csv"),
    cleaner=cleaner,
    validator=validator,
    judge=judge,
    output_path=workdir / "chatty_processed.csv",
    show_progress=False,
)
processed = pipeline.run()
print(list(processed.columns[-4:]))
shown = ["instruction_item_id", "profile_id", "random_seed", "answer", "valid", "decision"]
processed[shown].head(10)

['cleaned_answer', 'validation_status', 'valid', 'decision']


,instruction_item_id,profile_id,random_seed,answer,valid,decision
0,0,Optimistic Persona,1,1,True,1. Disagree strongly
1,0,Conservative Persona,1,"{answer: ""1. Disagree strongly""}",True,1. Disagree strongly
2,1,Optimistic Persona,1,5,True,5. Agree strongly
3,1,Conservative Persona,1,"Sorry, I can't answer that.",False,not present
4,2,Optimistic Persona,1,I would say Disagree a little.,True,2. Disagree a little
5,2,Conservative Persona,1,"{answer: ""2. Disagree a little""}",True,2. Disagree a little
6,3,Optimistic Persona,1,I would say Neither agree nor disagree.,True,3. Neither agree nor disagree
7,3,Conservative Persona,1,"As an AI, I do not have personal opinions.",False,not present
8,4,Optimistic Persona,1,2. Disagree a little,True,2. Disagree a little
9,4,Conservative Persona,1,"As an AI, I do not have personal opinions.",False,not present


In [19]:
processed["decision"].value_counts()

decision
not present                      13
2. Disagree a little              4
4. Agree a little                 4
1. Disagree strongly              3
5. Agree strongly                 3
3. Neither agree nor disagree     3
Name: count, dtype: int64

## 11. From answers to scores

The `weight` of the chosen answer option is the score of the answer. Two details of
questionnaire scoring are handled explicitly: **reversed items** (`reversed: true`) flip
the scale, and refusals or undecided answers are left out instead of being counted as
zero. Then the scores can be averaged per persona and dimension (`attributes` of the
items and the questionnaire name the dimensions).

In [20]:
def to_scores(experiment, processed):
    """Map the judged answer options to numbers (reversed items flipped, refusals dropped)."""
    options = experiment.questionnaire.default_answer_options.options.values()
    weights = {o.text: o.weight for o in options if not o.ignored_for_scale}
    low, high = min(weights.values()), max(weights.values())
    items = experiment.questionnaire.instruction_items

    scores = processed["decision"].map(weights).where(processed["valid"])
    reverse = processed["instruction_item_id"].map(lambda i: items[i].reversed)
    return scores.where(~reverse, low + high - scores)


processed["score"] = to_scores(experiment, processed)

names = experiment.questionnaire.attributes["dimension"]
items = experiment.questionnaire.instruction_items
processed["dimension"] = processed["instruction_item_id"].map(
    lambda i: names[items[i].attributes["dimension"]]
)
processed.groupby(["profile_id", "dimension"])["score"].agg(["count", "mean"]).round(2)

count  mean
profile_id           dimension                     
Conservative Persona Agreeableness          1  3.00
                     Conscientiousness      2  3.50
                     Extraversion           2  2.00
                     Neuroticism            0   NaN
                     Openness               2  3.00
Optimistic Persona   Agreeableness          1  1.00
                     Conscientiousness      3  3.33
                     Extraversion           1  1.00
                     Neuroticism            3  3.00
                     Openness               2  3.00

`count` is the number of answers that could be scored (at most 4 questions x 3 seeds),
the others were refusals or not recognisable. The demo questionnaire only contains
extraversion items; the full inventories in `examples/data/` cover all dimensions.

## 12. The command line

The same workflow is available from the shell as the `rupsycho` command (also
`python -m rupsycho`). These cells are not executed here, because `run` would load the
real model of the example:

```bash
rupsycho examples copy bfi bfi.json                  # start from the bundled example
rupsycho validate bfi.json                           # check the file, load no model
rupsycho prompt bfi.json --item 0 --persona 1        # print the assembled prompt
rupsycho run bfi.json --dry-run                      # number of calls and a prompt, no model
rupsycho run bfi.json -o results.csv --seeds 1 2 3 --max-concurrency 4
rupsycho postprocess bfi.json results.csv -o scored.csv --pattern 'answer\s*:\s*"([^"]+)"'
```

`run` streams the answers into the CSV file while it works and exits with status 3 if
some model calls failed (`--on-error` selects the policy of section 7). The
[command-line reference](https://julianschelb.github.io/rupsycho/cli/) lists all
options.

## Where next

* `examples/Basics.ipynb` and `examples/Callbacks.ipynb` run the demo configuration with
  a real Hugging Face model (they download one).
* `examples/Parsers.ipynb` and `examples/Postprocessing.ipynb` go through every cleaner,
  validator and judge and the post-processing pipeline.
* `examples/*.py` are runnable scripts: a real run, a reproducibility check, a custom
  SQLite callback and configuration snippets for OpenAI, Ollama, Google and DeepSeek.
* The documentation explains the concepts and every configuration field:
  <https://julianschelb.github.io/rupsycho/>.

Finally, remove the scratch folder:

In [21]:
import shutil

shutil.rmtree(workdir)
workdir.exists()

False